# Executar pipeline e subir o Streamlit

Use este notebook quando quiser rodar o fluxo inteiro no ambiente local:

1. executar os notebooks 01 e 02;
2. conferir os artefatos principais;
3. iniciar o Streamlit.

In [1]:


import sys
from pathlib import Path


DIRETORIO_NOTEBOOKS = Path.cwd()
if not (DIRETORIO_NOTEBOOKS / "ambiente_notebook.py").exists():
    candidato = DIRETORIO_NOTEBOOKS / "notebooks"
    if candidato.exists():
        DIRETORIO_NOTEBOOKS = candidato
if str(DIRETORIO_NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(DIRETORIO_NOTEBOOKS))

from ambiente_notebook import adicionar_raiz_no_syspath

DIRETORIO_RAIZ = adicionar_raiz_no_syspath()

import subprocess
import sys
import time
import urllib.request
import webbrowser

import nbformat as nbf
import pandas as pd
from nbconvert.preprocessors import ExecutePreprocessor

NOTEBOOKS_FLUXO = [
    DIRETORIO_RAIZ / "notebooks" / "01_tratamento_e_bases_modelo.ipynb",
    DIRETORIO_RAIZ / "notebooks" / "02_painel_analitico.ipynb",
]

PROCESSO_STREAMLIT = None

In [2]:
def executar_notebook(caminho_notebook: Path) -> Path:
    with caminho_notebook.open("r", encoding="utf-8") as arquivo:
        notebook = nbf.read(arquivo, as_version=4)

    executor = ExecutePreprocessor(timeout=900, kernel_name="python3")
    executor.preprocess(notebook, {"metadata": {"path": str(caminho_notebook.parent)}})

    with caminho_notebook.open("w", encoding="utf-8") as arquivo:
        nbf.write(notebook, arquivo)

    return caminho_notebook


def executar_fluxo_notebooks() -> pd.DataFrame:
    linhas = []
    for caminho_notebook in NOTEBOOKS_FLUXO:
        inicio_execucao = time.time()
        executar_notebook(caminho_notebook)
        linhas.append(
            {
                "notebook": caminho_notebook.name,
                "status": "executado",
                "duracao_seg": round(time.time() - inicio_execucao, 1),
            }
        )
    return pd.DataFrame(linhas)


def aguardar_streamlit(url_base: str, tentativas: int = 20, pausa_seg: float = 1.0) -> bool:
    for _ in range(tentativas):
        try:
            with urllib.request.urlopen(url_base, timeout=2):
                return True
        except Exception:
            time.sleep(pausa_seg)
    return False


def iniciar_streamlit(
    porta: int = 8501,
    modo_headless: bool = True,
    abrir_navegador: bool = True,
) -> int:
    global PROCESSO_STREAMLIT
    if PROCESSO_STREAMLIT is not None and PROCESSO_STREAMLIT.poll() is None:
        print(f"Streamlit já está rodando no PID {PROCESSO_STREAMLIT.pid}.")
        return PROCESSO_STREAMLIT.pid

    caminho_log = DIRETORIO_RAIZ / "artifacts" / "logs" / "streamlit_runner.log"
    caminho_log.parent.mkdir(parents=True, exist_ok=True)
    arquivo_log = caminho_log.open("w", encoding="utf-8")
    comando = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        str(porta),
        "--server.headless",
        "true" if modo_headless else "false",
    ]
    PROCESSO_STREAMLIT = subprocess.Popen(
        comando,
        cwd=str(DIRETORIO_RAIZ),
        stdout=arquivo_log,
        stderr=subprocess.STDOUT,
    )

    url = f"http://localhost:{porta}"
    if aguardar_streamlit(url):
        if abrir_navegador:
            webbrowser.open_new_tab(url)
        print(f"Streamlit iniciado. PID={PROCESSO_STREAMLIT.pid} | URL={url}")
    else:
        print("O Streamlit foi iniciado, mas a URL não respondeu no tempo esperado.")

    print(f"Log: {caminho_log}")
    return PROCESSO_STREAMLIT.pid


def encerrar_streamlit() -> None:
    global PROCESSO_STREAMLIT
    if PROCESSO_STREAMLIT is None or PROCESSO_STREAMLIT.poll() is not None:
        print("Não há processo ativo do Streamlit.")
        return
    PROCESSO_STREAMLIT.terminate()
    PROCESSO_STREAMLIT.wait(timeout=15)
    print("Streamlit finalizado.")
    PROCESSO_STREAMLIT = None

## 1. Executar os notebooks

In [3]:
relatorio_execucao = executar_fluxo_notebooks()
relatorio_execucao

,notebook,status,duracao_seg
0,01_tratamento_e_bases_modelo.ipynb,executado,31.7
1,02_painel_analitico.ipynb,executado,6.7


## 2. Conferir os artefatos

In [4]:
artefatos_principais = [
    DIRETORIO_RAIZ / "data" / "processed" / "base_analitica.parquet",
    DIRETORIO_RAIZ / "data" / "processed" / "base_modelagem_proximo_ano.parquet",
    DIRETORIO_RAIZ / "artifacts" / "model" / "model_pipeline.joblib",
    DIRETORIO_RAIZ / "artifacts" / "analytics" / "painel_analitico.json",
]

quadro_validacao = pd.DataFrame(
    [
        {
            "arquivo": caminho.name,
            "existe": caminho.exists(),
            "caminho": str(caminho),
        }
        for caminho in artefatos_principais
    ]
)
quadro_validacao

,arquivo,existe,caminho
0,base_analitica.parquet,True,C:\Users\murilo.santos\Documents\tech5challeng...
1,base_modelagem_proximo_ano.parquet,True,C:\Users\murilo.santos\Documents\tech5challeng...
2,model_pipeline.joblib,True,C:\Users\murilo.santos\Documents\tech5challeng...
3,painel_analitico.json,True,C:\Users\murilo.santos\Documents\tech5challeng...


## 3. Iniciar o Streamlit

A célula abaixo inicia o app local e tenta abrir o navegador quando a URL estiver pronta.

In [5]:
iniciar_streamlit(porta=8501, modo_headless=True, abrir_navegador=True)

Streamlit iniciado. PID=51564 | URL=http://localhost:8501
Log: C:\Users\murilo.santos\Documents\tech5challenge\artifacts\logs\streamlit_runner.log


51564

## 4. Encerrar o Streamlit

In [6]:
# Execute esta célula apenas quando quiser encerrar o app.
# encerrar_streamlit()